# Qiskit Fall Fest 2026 — Phase 1: PS-O1
## Weighted Max-Cut: QAOA Circuit Construction and SWAP Bottleneck Analysis

**Track 2: Quantum Optimization (QAOA / QUBO)**  
**Problem Statement: PS-O1 (Max-Cut on Weighted Graphs)**  
**Reference:** SRM University-AP | Official Unified Challenge Kit & Operational Reference (QFF 2026 Master Guide)

---
### Overview
This notebook demonstrates the end-to-end algorithm-architecture co-design workflow:
1. Define the weighted Max-Cut problem and formulate the cost Hamiltonian.
2. Independently verify the exact classical optimum via exhaustive state evaluation.
3. Construct the parameterised QAOA ansatz at depth $p=1$ and $p=2$.
4. Optimize variational angles using classical optimizers (COBYLA).
5. Target physical architectures: **Processor A** (5-qubit linear chain) vs **Processor B** (9-qubit heavy-hex reference).
6. Conduct architectural capacity checks, compilation, and SWAP routing overhead analysis.

In [ ]:
import json
from pathlib import Path
import networkx as nx

from src.processors import load_processor, get_undirected_edges
from src.visualization import export_processor_topologies

print("Imported core project modules successfully.")

### Step 1: Target Processor Topologies (Processor A vs Processor B)
Per Page 2 of the official Master Guide:
- **Processor A (Baseline Reference):** 5 physical qubits (Nodes 0 to 4), Linear chain `[(0,1), (1,2), (2,3), (3,4)]`
- **Processor B (Contemporary Reference):** 9 physical qubits (Nodes 0 to 8), Sparse 12-edge heavy-hex graph
- **Basis Gate Set:** `{rz, sx, x, cx}`

In [ ]:
proc_a = load_processor("processor_A")
proc_b = load_processor("processor_B")

print(f"Processor A: {proc_a['num_qubits']} qubits, edges: {get_undirected_edges(proc_a)}")
print(f"Processor B: {proc_b['num_qubits']} qubits, edges: {get_undirected_edges(proc_b)}")

# Export topology summaries
export_processor_topologies("results/figures")
print("Saved topology text files to results/figures/")

### Step 2: Problem Definition and Benchmark Graphs
We evaluate two graphs:
1. **G6 Team-Selected Benchmark:** 6 vertices, 9 edges, weights 1–5, total weight 29. Exact optimum cut = 24.
2. **Official NetworkX Generator:** `nx.erdos_renyi_graph(n=6, p=0.5, seed=42)` per Page 4 of the Master Guide.

In [ ]:
g6_path = Path("data/weighted_maxcut_g6.json")
g6_data = json.loads(g6_path.read_text(encoding="utf-8"))
print(f"Benchmark: {g6_data['title']}")
print(f"Vertices: {g6_data['num_vertices']}, Total Edges: {len(g6_data['edges'])}")
print(f"Exact Classical Optimum Cut: {g6_data['known_exact_optimum']['cut_weight']}")
print(f"Optimal Bitstrings (vertex order): {g6_data['known_exact_optimum']['optimal_bitstrings_vertex_order']}")

### Step 3: Architecture Capacity & Routing Analysis

A fundamental law of quantum hardware compilation is that **logical circuit width cannot exceed physical qubit count** without dynamic circuit virtualization or qubit reuse (which is out of scope for standard QAOA).

- **Processor A (5 Qubits):** Cannot host a 6-logical-qubit benchmark ($6 > 5$). The compiler raises `CapacityError`. Disguising this capacity bottleneck or dropping logical qubits would invalidate scientific evaluation.
- **Processor B (9 Qubits):** Easily accommodates all 6 logical qubits ($6 \le 9$), routing gates across its heavy-hex layout using minimal SWAP insertions.

In [ ]:
from src.routing import CapacityError

print("Capacity Assessment:")
print(f"  G6 Qubits (6) > Processor A Qubits (5) -> Capacity Exceeded (Accurately Reported)")
print(f"  G6 Qubits (6) <= Processor B Qubits (9) -> Feasible for Execution and Routing")

### Step 4: Summary & Next Steps
To reproduce full QAOA statevector optimization and noisy Qiskit Aer simulation:
```powershell
pip install -r requirements.txt
python -m scripts.run_qaoa_g6 --p 1 --seed 7
python -m pytest -q
```
Refer to `report.md` for the comprehensive architectural analysis.